# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a structured guide for loading and exploring the FAIR² mass spectrometry dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL (Croissant schema)
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Get record sets
record_sets = metadata.record_sets
if not record_sets:
    print("No record sets found in the metadata. The dataset may reference record sets via file distributions.")
else:
    print(f"Found {len(record_sets)} record set(s):\n")
    for rs in record_sets:
        print(f"- RecordSet @id: {rs['@id']} Name: {rs.get('name', '')}")

# You can also inspect distributions for data files if record_sets is empty
if hasattr(metadata, 'distributions') and metadata.distributions:
    print("\nDistributions (Data Files) in Metadata:")
    for dist in metadata.distributions:
        print(f"- Distribution @id: {dist.get('@id')} | Content URL: {dist.get('contentUrl')}")

# Example: Peek at available fields/columns if present
if record_sets:
    # Take the first record set for exploration
    rs0 = record_sets[0]
    fields = rs0.get('field', [])
    print(f"\nFields in RecordSet {rs0['@id']}:")
    for f in fields:
        print(f"- Field @id: {f['@id']} Name: {f.get('name', '')} Type: {f.get('dataType', '')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# --- Identify available record set IDs for extraction ---
# If there are no record_sets, we assume a default (majority of Croissant datasets have at least one, else reference via distribution/file).

# Try to find the record set id dynamically
record_set_ids = [rs['@id'] for rs in metadata.record_sets] if metadata.record_sets else []

# For this dataset, we'll assume a record set @id, otherwise try to fall back to exploring records via available @id keys.

if not record_set_ids:
    # Try to infer recordSet id from file distribution or use Croissant standard
    # For FAIR², the primary record set id commonly starts with 'cr:RecordSet' or has a full URL.
    # Example: 'https://api.app.sen.science/frontiers/7154140/071e8e85-744b-4afd-941a-3777bffc2a09' (not provided directly in the schema stub above)
    print("No explicit record sets; cannot extract records.")
    dataframes = {}
else:
    dataframes = {}
    for record_set_id in record_set_ids:
        # Load records for each record set
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records for RecordSet: {record_set_id}")

    # Show sample columns and preview from the first record set
    main_rs = record_set_ids[0]
    print("Available columns for first record set:")
    print(dataframes[main_rs].columns.tolist())
    dataframes[main_rs].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering records based on specific criteria, normalizing numeric fields, and grouping data by key attributes to prepare it for further analysis.

In [ ]:
# --- Only proceed if we have a loaded DataFrame ---
if dataframes:
    # For this analysis, we'll assume common proteomics fields likely present based on the dataset's description.
    # Let's inspect the columns to select a numeric field and a grouping field.
    main_rs = list(dataframes.keys())[0]
    df = dataframes[main_rs]

    print(f"Columns: {df.columns.tolist()}")
    
    # Try to auto-select a numeric field and a group field
    # We'll guess common field ids based on proteomics, e.g., 'cr:PeptideCount', 'cr:MW_kDa', etc.
    # Replace below with actual @id or column name as needed.
    numeric_field = None
    group_field = None
    
    numeric_candidates = [col for col in df.columns if ('count' in col.lower() or 'abundance' in col.lower() or 'mw' in col.lower() or 'coverage' in col.lower())]
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
    group_candidates = [col for col in df.columns if ('description' in col.lower() or 'gene' in col.lower() or 'sample' in col.lower())]
    if group_candidates:
        group_field = group_candidates[0]

    print(f"Selected numeric field: {numeric_field}")
    print(f"Selected group field: {group_field}")

    if numeric_field is not None and df[numeric_field].dtype != 'O':
        # Perform filtering
        threshold = df[numeric_field].mean()
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        print(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        if group_field and group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"\nGrouped data by {group_field} (mean):")
            print(grouped_df.head())
    else:
        print("No numeric field available for filtering and normalization.")
else:
    print("No DataFrame available.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field is not None and numeric_field in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Histogram of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    # If group_field exists, show boxplot
    if group_field and group_field in df.columns:
        plt.figure(figsize=(12, 6))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.xticks(rotation=45)
        plt.title(f"{numeric_field} by {group_field}")
        plt.show()
else:
    print("No suitable DataFrame or numeric field for visualization.")

## 6. Conclusion
This notebook demonstrated how to load and explore complex tabular datasets described with the Croissant FAIR schema using the `mlcroissant` library. We examined dataset metadata, loaded record sets, performed basic exploratory data analysis, and visualized distributions. This process can be extended for deeper domain-specific analyses and model training.